RMRB Analysis v4
- (2024.09.16) Build `Extract_Ad_Block`, major in extracting ad blocks from original images
- Improve `CV_Detect_Ads` to fit `Extract_Ad_Block` function
- (2024.09.17) Because there are a lot of duplicated images after using CV block conversion. For the convenience of solving this problem, we change the named system for ad images use `Change_Named_System`. The rule of changing show as follows:
  - Full-page ad images use "FAD" symbol. For example: 20220101_8_FAD.png
  - Half-page dectected with ciphertext ad images use "HAD" symbol. Like: 20220104_13_HAD.png
  - Half-page dectected with CV ad images keep the prototype. Like: 20220104_16_CV.png
- After changing the named system. We can solve this problems accroding to image types use `Check_Duplicated_Images`:
  - For "FAD": Do not use CV ad block conversion
  - (2024.09.18) For "HAD" and "CV": we would like to analysis shape of an ad block to ensure extract the ad block more precisely. The analysis modular is integrated in `Extract_Ad_Block` and the major component is in `Ad_Shape_Analysis`.
    - Ad Shape Analysis perspectives:
      - 1. w, h, and w / h for all true ad blocks
      - 2. For a filter condition, how accurate is it?
  - After filtering: Store all image paths which meet the criteria in a pkl file every year (Use pickle format).
    - Shape list (All ad blocks): `f'{YEAR}_Shape_List.pkl'`
    - Filter list (All correct ad blocks from Shape list): `f'{YEAR}_Filter_List.pkl'`
    - Outlier list (All incorrect ad blocks from Shape list): `f'{YEAR}_Outlier_List.pkl'`
  - (2024.09.23) Add a function in `Check_Duplicated_Images`: we can choose the image path set either from a whole year's worth of files, or more precisely, from `f'{YEAR}_Filter_List.pkl'`.
  - Finally: we should remove duplicated files in `f'{YEAR}_Filter_List.pkl'`. Most the duplicated images in `f'{YEAR}_Filter_List.pkl'` are all ad blocks, there's just a difference in the way it's cut, resulting in inconsistent picture sizes. We can choose the largest images with `Compare_File_Sizes` for each duplicated files group and stored in `Transfered_Paths`. Ultimately, we remove those image paths from `f'{YEAR}_Filter_List.pkl'` and added in `f'{YEAR}_Outlier_List.pkl'`. We don't modify original files. New files are created as `f'{YEAR}_Final_Filter.pkl'` and `f'{YEAR}_Final_Outlier.pkl'`.
- Text recognition with `Text_Recognition`: After finishing all above ad images extraction tasks. We can simply use OCR to recognize ad text for images which in `f'{YEAR}_Final_Filter.pkl'` and "FAD" type images.
- (2024.09.19~2024.09.20) Text summarization `Summarized_Text`: We would like to use LLM to summarize the text.
  - Additional packages: `jieba`, `pytorch_lighting`, `rouge`, `fengshen`(from github), `transformers`
  - Some characters may be recognized as a special text. We would like to use `Modify_Chars` to modify.
  - (2024.09.24) Improve `Summarized_Text`: Add new pares: `Model_Name`, `Max_Len`: Max length of the model, `Text_Type`: Type of OCR, `Text_Language`: Language of OCR recognition.
- (2024.09.23) We should store the original text and summary text in files. After consideration, we use json file, which can be easily modified and saved.
- (2024.09.24) For displaying output convenience, we studied the display system of Jupyter IDE (Show in [HTML_In_Python](../Test/HTML_In_Python.ipynb)). Now we would like to improve our output content as following:
  - For `.py` file: use original output approach (main in `print()`)
  - For Jupyter IDE (`.ipynb` file): use `display()` fun to show output as it can create a clickable hyperlink by using the HTML capabilities of IPython display.
    - For example: This is one of an `Summarized_Text` output: 
      ```python
      print(f"{Text_Type}_{Lan} summarized by {Model_Name}_m{Max_Len} to {json_path}")
      ```
      > Output:
      >
      > Pytesseract_Text_C summarized by RMRB_model_2_m1024 to [D:/AI_data_analysis/RMRB/2022_AD/20221230_8_FAD.json](D:/AI_data_analysis/RMRB/2022_AD/20221230_8_FAD.json)
    - As we wish to access its original image immediately, we can add image path after json path. However, it results in extremely long text. Therefore, we would like to take advantages of the HTML feature in Jupyter. Thus, we can achieve it as following:

      ```python
      from IPython.display import HTML, display
      image_path = "D:/AI_data_analysis/RMRB/2022_AD/20221230_8_FAD.png"
      # Create a clickable link, which named "Image_Link"
      link = f'<a href="file:///{image_path}" target="_blank">Image_Link</a>'
      output = f"{Text_Type}_{Lan} summarized by {Model_Name}_m{Max_Len} to {json_path} ({link})"
      display(output)
      ```
      > Output:
      >
      > Pytesseract_Text_C summarized by RMRB_model_2_m1024 to [D:/AI_data_analysis/RMRB/2022_AD/20221230_8_FAD.json](D:/AI_data_analysis/RMRB/2022_AD/20221230_8_FAD.json) ([Image_Link](D:/AI_data_analysis/RMRB/2022_AD/20221230_8_FAD.png))

In [1]:
from PyPDF2 import PdfReader, PdfWriter
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from datetime import datetime, timedelta
from pathlib import Path
import os
import cv2
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from collections import defaultdict
from PIL import Image
import warnings
warnings.filterwarnings("ignore")
import json
import pickle

# from pdfminer.pdfdocument import PDFDocument
# from pdfminer.pdfparser import PDFParser
# from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
# from pdfminer.converter import PDFPageAggregator
# from pdfminer.layout import LTTextBoxHorizontal, LAParams
# from pdfminer.high_level import extract_text
# from pdfminer.pdfinterp import PDFTextExtractionNotAllowed

from transformers import (
    pipeline, 
    PegasusForConditionalGeneration, 
    PegasusForConditionalGeneration
)
from tokenizers_pegasus import PegasusTokenizer

# Set the path to the Tesseract installation
pytesseract.pytesseract.tesseract_cmd = "D:/Tesseract-OCR/tesseract.exe"

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\HH/.cache/jieba.cache
Loading model from cache C:\Users\HH/.cache/jieba.cache
Loading model cost 0.625 seconds.
Prefix dict has been built successfully.


In [2]:
Advertisement = "广告"
Cipher_AD = """!!"!"""

In [3]:
ROOT_PATH = "D:/AI_data_analysis/RMRB/"
RMRB2024090616 = "D:/AI_data_analysis/RMRB/2024090616.pdf"
def PATH_FUN(YEAR, MONTH, DAY, EDITION, SUFFIX, NAME_bool=False):
    Split_Year = ["2023"] 
    NAME = YEAR + MONTH + DAY
    INFO_PATH = (
        NAME 
        + (
            ("/" + NAME + EDITION) 
            if YEAR in Split_Year else ""
        ))
    SUFFIX = ".pdf"
    pdf_path = ROOT_PATH + YEAR + "/" + INFO_PATH + SUFFIX
    if NAME_bool:
        return pdf_path, NAME
    else: return  pdf_path
PDF_PATH = PATH_FUN("2022", "01", "02", "02", ".pdf")
PDF_PATH

'D:/AI_data_analysis/RMRB/2022/20220102.pdf'

In [4]:
def Generate_Dates(year):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year + 1, 1, 1)
    current_date = start_date
    
    dates = []
    while current_date < end_date:
        dates.append(current_date.strftime('%Y%m%d'))
        current_date += timedelta(days=1)
    return dates

def Check_PDF_Exist(YEAR, Begin_date="0101", End_date="1231"):
    MISSING_PDF = []
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    current_date = start_date
    while current_date <= end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PATH = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf")
        # try:
        #     with open(PATH, "rb") as file:
        #         PdfReader(file)
        # except FileNotFoundError: 
        #     print(PATH)
        #     MISSING_PDF.append(PATH)
        # print(PATH)
        if not os.path.isfile(PATH):
            MISSING_PDF.append(PATH)
        current_date += timedelta(days=1)
    return MISSING_PDF
# Check_PDF_Exist("2016")

In [5]:
def Count_Files(folder_path):
    # Create a Path object for the folder
    folder = Path(folder_path)
    # Count all files in the folder
    file_count = sum(1 for file in folder.iterdir() if file.is_file())
    return file_count
# Count_Files("D:/AI_data_analysis/RMRB")

def Check_Folder_Exist(folder_path):
    # Check if the folder exists
    if not os.path.exists(folder_path):
        # Create the folder if it does not exist
        os.makedirs(folder_path)
        print(f'Folder created: {folder_path}')
    else:
        print(f'Folder already exists: {folder_path}')
# Check_Folder_Exist("D:/AI_data_analysis/RMRB")

In [6]:
def PDF2Image(pdf_path: str, actual_page_list):
    PDF = PdfReader(pdf_path)
    root_path = "/".join(pdf_path.split("/")[:-1]) + "/"
    Name = pdf_path.split("/")[-1].split(".")[0]
    num_of_pages = len(PDF.pages)
    print(f"PDF page(s) num: {num_of_pages}" + "\n")
    print("PDF PATH:", PDF_PATH)
    # Convert PDF pages to images
    page_list = [num - 1 for num in actual_page_list]

    for page_num in page_list:
        IMAGE = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
        page_num_str = ("0" + str(page_num + 1)) if len(str(page_num + 1)) == 1 else str(page_num + 1)
        image_path = root_path + Name + f"{page_num_str}.png"
        IMAGE[0].save(image_path, "PNG")
        print(f"Page {page_num + 1} image saved to: {image_path}")
# PDF2Image("D:/AI_data_analysis/RMRB/20200117.pdf", [5, 12, 18, 20])

In [7]:
def CV_Detect_Ads(
        root_path: str,
        image_type: str="CV",
        pdf_name: str='', 
        pdf_version: str='1', 
        image_path: str='',
        image_element=None, 
        Threshold: list=[0.4, 0.6], 
        Image_Path_Bool: bool=False,
        Image_Clip_Bool: bool=False,
        Whole_Image_Bool: bool=True,
        AD_SHAPE_ANALYSIS: bool=False
    ):
    """
    image_type: {"FAD", "HAD", "CV"}

    pdf_name: pdf name

    pdf_version: version of a pdf for image conversion, begin with '1'

    image_element: the image element which generated from `convert_from_path()`

    Threshold list: [min, max] (area thershold is [min * whole_area, max * whole_area])

    image_path_bool: if it is True, use image path, or use others path (like pdf path)

    Image_Clip_Bool: Whether clip the detected image to an individual image. Default False.

    Whole_Image_Bool: Whether save the marked image
    """
    if Image_Path_Bool:
        # Load the image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Failed to load image: {image_path}")
            return False
    else:
        # Convert Pillow image to numpy array
        image_np = np.array(image_element)
        # Convert RGB to BGR since OpenCV uses BGR format
        image = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    if AD_SHAPE_ANALYSIS: SHAPE_list = []

    # Get the dimensions of the image
    height, width = image.shape[:2]
    # Calculate the area of the entire image
    image_area = height * width
    Ad_Block_Threshold_Min = Threshold[0] * image_area
    Ad_Block_Threshold_Max = Threshold[1] * image_area

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Apply edge detection
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ad_found = False  # Flag to indicate if an ad is detected

    i = 1
    # Iterate through contours
    for contour in contours:
        # print(contour)
        # Get the bounding box of the contour
        x, y, w, h = cv2.boundingRect(contour)
        # Calculate the area of the bounding rectangle
        area_rect = w * h # w is length, h is height

        # Filter based on area and aspect ratio (adjust thresholds as needed)
        if (area_rect >= Ad_Block_Threshold_Min) and (area_rect <= Ad_Block_Threshold_Max):
            ad_found = True

            if Whole_Image_Bool:
                # Draw a rectangle around the detected ad block
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
            if Image_Clip_Bool: 
                # Save the ad block image to local
                ad_block = image[y:y + h, x:x + w] # Clip the ad block
                output_block_path = root_path + pdf_name + "_" + pdf_version + f"_{image_type}_Block_{i}.png"
                cv2.imwrite(output_block_path, ad_block)
                if AD_SHAPE_ANALYSIS: 
                    TEXT = f"{pdf_version}_{i} of {output_block_path} with (w, h) = ({w}, {h}) (w / h = {w / h :.3f})"
                    print(TEXT)
                    SHAPE_list.append(TEXT)
                else: print(f"Version {pdf_version}_{i} Saved ad block to {output_block_path}")
                i += 1
    # Display the result or save it for later review
    if ad_found and Whole_Image_Bool:
        output_path = root_path + pdf_name + "_" + pdf_version + "_CV.png"
        cv2.imwrite(output_path, image)  # Save image with detected ads
        print(f"Version {pdf_version} Saved whole image to {output_path}")
    if AD_SHAPE_ANALYSIS: return SHAPE_list
    else: return None
# CV_Detect_Ads("D:/AI_data_analysis/RMRB/", "D:/AI_data_analysis/RMRB/20200117.pdf", "12", IMAGE_TEST[11])

In [8]:
# FOLD_PATH D:/AI_data_analysis/RMRB/2022_AD/
# TYPE HAD
# NAME 20220104_13_HAD
# VERSION 13
# IMAGE_PATH D:/AI_data_analysis/RMRB/2022_AD/20220104_13_HAD.png

In [9]:
def Analysis_AD_Position(YEAR, Begin_date="0101", End_date="1231"):
    Position_Dict = {}
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    print(f"Begin date: {YEAR + Begin_date}, End date: {YEAR + End_date}")
    current_date = start_date
    while current_date <= end_date:
        DATE = current_date.strftime("%Y%m%d")
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PDF_PATH = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf", NAME_bool=False)
        temp_dict = {}
        version_list = []
        with pdfplumber.open(PDF_PATH) as PDF:
            # PDF = PdfReader(file)
            PAGES_NUM = len(PDF.pages)
            for page_num in range(PAGES_NUM):
                text = PDF.pages[page_num].extract_text().replace(" ", "")
                if Advertisement in text:
                    Position = text.find(Advertisement)
                    DICT = {(page_num+1): Position}
                    version_list.append(DICT)
            temp_dict[DATE] = version_list
            print(temp_dict)
            Position_Dict[DATE] = version_list
        current_date += timedelta(days=1)
    
    # Initialize a Counter to keep track of version distributions
    version_distribution = Counter()

    # Iterate over each date and its corresponding list of dictionaries
    for date, versions in Position_Dict.items():
        for version_dict in versions:
            for version, count in version_dict.items():
                version_distribution[count] += 1
    # Prepare data for histogram
    # Get the count values for histogram
    counts = list(version_distribution.elements())

    # Plotting the histogram
    plt.figure(figsize=(10, 6))
    plt.hist(counts, bins=30, color='skyblue', alpha=0.7, edgecolor='black')
    plt.xlabel('Counts')
    plt.ylabel('Frequency')
    plt.title('Histogram of Version Counts')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
# Analysis_AD_Position("2022", Begin_date="1024")

In [10]:
def Remove_File_If_Exists(file_path):
    if os.path.isfile(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")

In [11]:
# Generate image for all ad pages using OCR
def Genetare_AD_Image(YEAR, Begin_date="0101", End_date="1231", Text_Range=34):
    PDF_CHECK = Check_PDF_Exist(YEAR, Begin_date, End_date)
    if PDF_CHECK: 
        print("PDF lackage!", PDF_CHECK)
        return
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    Check_Folder_Exist(FOLD_PATH)
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    print(f"Begin date: {YEAR + Begin_date}, End date: {YEAR + End_date}")
    current_date = start_date
    while current_date <= end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PDF_PATH, NAME = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf", NAME_bool=True)
        print("PDF PATH:", PDF_PATH)
        with pdfplumber.open(PDF_PATH) as PDF:
            # PDF = PdfReader(file)
            PAGES_NUM = len(PDF.pages)
            IMAGE = convert_from_path(PDF_PATH)
            for page_num in range(PAGES_NUM):
                text_original = PDF.pages[page_num].extract_text().replace(" ", "")
                text = text_original[:Text_Range+1]
                if Advertisement in text: # First filter: plaintext "广告"
                    # Attension: first_page and last_page starts at 1
                    IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_FAD.png"
                    IMAGE[page_num].save(IMAGE_PATH, "PNG")
                    print("Full-page Ad:", IMAGE_PATH)
                # elif Advertisement in text_original:
                #     Remove_File_If_Exists(IMAGE_PATH)
                elif """!!"!""" in text_original: # Second filter: ciphertext "广告"
                    # Attension: first_page and last_page starts at 1
                    # IMAGE = convert_from_path(PDF_PATH, first_page=page_num+1, last_page=page_num+1)
                    IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_HAD.png"
                    IMAGE[page_num].save(IMAGE_PATH, "PNG")
                    print("Half-page Ad", IMAGE_PATH)
                else:
                    CV_Detect_Ads(root_path=FOLD_PATH, pdf_name=NAME, 
                                  pdf_version=str(page_num+1), 
                                  image_element=IMAGE[page_num])
                    pass
        current_date += timedelta(days=1)
# Genetare_AD_Image("2015")
# PDF lackage! ['D:/AI_data_analysis/RMRB/2015/20150307.pdf']

In [12]:
# Function to rename a file
def Rename_File(current_path, new_path, Type="AD"):
    try:
        # Renaming the file
        os.rename(current_path, new_path)
        print(f"{Type}: Old File {current_path} renamed to: {new_path}")
    except FileNotFoundError:
        print(f"Error: The file {current_path} does not exist.")
    except PermissionError:
        print(f"Error: You do not have permission to rename {current_path}.")
    except Exception as e:
        print(f"Unexpected error: {e}")

def Change_Named_System(YEAR, Begin_date="0101", End_date="1231", Text_Range=34):
    PDF_CHECK = Check_PDF_Exist(YEAR, Begin_date, End_date)
    if PDF_CHECK: 
        print("PDF lackage!", PDF_CHECK)
        return
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    Check_Folder_Exist(FOLD_PATH)
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    print(f"Begin date: {YEAR + Begin_date}, End date: {YEAR + End_date}")
    current_date = start_date
    while current_date <= end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PDF_PATH, NAME = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf", NAME_bool=True)
        print("PDF PATH:", PDF_PATH)
        with pdfplumber.open(PDF_PATH) as PDF:
            # PDF = PdfReader(file)
            PAGES_NUM = len(PDF.pages)
            for page_num in range(PAGES_NUM):
                text_original = PDF.pages[page_num].extract_text().replace(" ", "")
                text = text_original[:Text_Range+1]
                if Advertisement in text: # First filter: plaintext "广告"
                    OLD_IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_AD.png"
                    NEW_IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_FAD.png"
                    Rename_File(OLD_IMAGE_PATH, NEW_IMAGE_PATH, "FAD")
                elif """!!"!""" in text_original: # Second filter: ciphertext "广告"
                    OLD_IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_AD.png"
                    NEW_IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_HAD.png"
                    Rename_File(OLD_IMAGE_PATH, NEW_IMAGE_PATH, "HAD")
        current_date += timedelta(days=1)
# Change_Named_System("2020")
# Old named system: No

In [13]:
def Extract_Ad_Block(
    YEAR, Begin_date="0101", End_date="1231", 
    Ad_Shape_Analysis: bool = True
):
    """
    All image names are formated like this: YYYYMMDD_Version_Type.png
    Type is in {FAD, HAD, CV}
    For example: 20220101_8_FAD.png, 20220104_16_CV.png

    Ad block extraction rules:
    1. For "FAD"
    """
    if Ad_Shape_Analysis: SHAPE_LIST = []
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    print(f"Begin date: {YEAR + Begin_date}, End date: {YEAR + End_date}")
    filtered_image_names = []
    for filename in os.listdir(FOLD_PATH):
        file_path = os.path.join(FOLD_PATH, filename)
        name = filename.split(".")[0]
        suffix = filename.split(".")[1]
        # Check if it is a file (not a directory)
        if os.path.isfile(file_path):  # Ensure it is a file
            if suffix == "png": # Ensure it is an image
                if (
                    len(name.split("_")) == 3
                ):  # Ensure the image is original (Avoid repeated extraction)
                    if name.split("_")[2] != "FAD":  # "FAD" ads don't need to extract
                        # Attention that the file names include suffix like '20220104_13_HAD.png'
                        date_str = name.split("_")[0]  # Get the date part
                        file_date = datetime.strptime(
                            date_str, "%Y%m%d")  # Convert to datetime object
                        # Check if the file date is within the specified range
                        if start_date <= file_date <= end_date:
                            filtered_image_names.append(filename)
    DATE_i= filtered_image_names[0].split("_")[0]
    print(DATE_i)
    for image_name in filtered_image_names:
        IMAGE_NAME = image_name.split(".")[0]
        PDF_NAME = IMAGE_NAME.split("_")[0]
        IMAGE_PATH = os.path.join(FOLD_PATH, image_name)
        DATE = IMAGE_NAME.split("_")[0]
        if DATE != DATE_i:  # This is used for date print
            print(DATE)
            DATE_i = DATE
        VERSION = IMAGE_NAME.split("_")[1]
        TYPE = IMAGE_NAME.split("_")[2]
        # For result correction, we use original thershold [0.4, 0.6]
        Shape_list = CV_Detect_Ads(
            root_path=FOLD_PATH,
            image_type=TYPE,
            pdf_name=PDF_NAME,
            pdf_version=VERSION,
            image_path=IMAGE_PATH,
            Image_Path_Bool=True,
            Image_Clip_Bool=True,
            Whole_Image_Bool=False,
            AD_SHAPE_ANALYSIS=Ad_Shape_Analysis,
        )
        if Ad_Shape_Analysis: SHAPE_LIST.append(Shape_list)
    if Ad_Shape_Analysis: 
        # Saving to a pickle file
        with open(ROOT_PATH + f"{YEAR}_AD/" + f"{YEAR}_Shape_List.pkl", 'wb') as f:
            pickle.dump(Shape_list, f)
        return SHAPE_LIST
    else: return None

# Shape_list = Extract_Ad_Block("2022", Ad_Shape_Analysis=True)
# 2022: 12 mins

In [14]:
def Ad_Shape_Analysis(YEAR):
    def Filter(W, H, W_Divide_H):
        if 2770 <= int(W)  and int(w) <= 2790:
            if int(H):
                if float(W_Divide_H):
                    return True
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    Shape_List_Path = FOLD_PATH + f'{YEAR}_Shape_List.pkl'
    if os.path.exists(Shape_List_Path):
        # Loading back the shape list if it exists
        with open(Shape_List_Path, 'rb') as f:
            Shape_List = pickle.load(f)
    else: 
        print(f"{Shape_List_Path} does not exist.")
        return

    Filter_Num = 0
    Outlier_Num = 0
    Filter_list = []
    Outlier_list = []
    Outlier_list_With_Shape = []
    for shape_list in Shape_List:
        for shape in shape_list:
            shape_split = shape.split(" ")
            link = shape_split[2]
            w = shape_split[7][1:-1]
            h = shape_split[8][:-1]
            w_divide_h = shape_split[13][:-1]
            if Filter(w, h, w_divide_h):
                print(link, w, h, w_divide_h)
                Filter_list.append(link)
                Filter_Num += 1
            else: 
                Outlier_list_With_Shape.append([link, w, h, w_divide_h])
                Outlier_list.append(link)
                Outlier_Num += 1
    print("Filter num:", Filter_Num)
    print("*" * 80)
    for outlier_list in Outlier_list_With_Shape:
        print(outlier_list[0], outlier_list[1], outlier_list[2], outlier_list[3])
    print("Outlier num:", Outlier_Num)
    print("*" * 80)

    # Saving Filter list to a pickle file
    with open(FOLD_PATH + f'{YEAR}_Filter_List.pkl', 'wb') as f:
        pickle.dump(Filter_list, f)
        print(f"{FOLD_PATH + f'{YEAR}_Filter_List.pkl'} saved.")

    # Saving Outlier list to a pickle file
    with open(FOLD_PATH + f'{YEAR}_Outlier_List.pkl', 'wb') as f:
        pickle.dump(Outlier_list, f)
        print(f"{FOLD_PATH + f'{YEAR}_Outlier_List.pkl'} saved.")
# Ad_Shape_Analysis("2022")

In [15]:
def Get_File_Size(file_path):
    """Returns the size of the file in bytes or -1 if the file does not exist."""
    if os.path.isfile(file_path):
        return os.path.getsize(file_path)
    else:
        print(f"File not found: {file_path}")
        return -1

def Compare_File_Sizes(file1, file2):
    """Compares the sizes of two files and returns the larger one."""
    size1 = Get_File_Size(file1)
    size2 = Get_File_Size(file2)

    if (size1 == -1) or (size2 == -1):
        return None  # Both files do not exist

    if size1 > size2:
        return file1
    elif size2 > size1:
        return file2
    else:
        return None  # Files are the same size
    
def Update_File_Lists(list_A, list_B, files_to_remove):
    """Updates list_A by removing specified files and adds them to list_B.
    
    Args:
        list_A (list): The list containing all files.
        list_B (list): The list to store removed files.
        files_to_remove (list): The list of files to be removed from list_A.
    """

    # Convert list_A to set for faster membership checking
    set_A = set(list_A)

    # Create a set from files_to_remove for quick access
    set_remove = set(files_to_remove)

    # Identify which files to remove that exist in list_A
    to_remove = set_A.intersection(set_remove)
    
    # Remove identified files and add them to list_B
    list_B.extend(to_remove)
    set_A.difference_update(to_remove)

    # Convert back to list if needed
    updated_list_A = list(set_A)
    updated_list_A.sort()
    list_B.sort()

    return updated_list_A, list_B

def Check_Duplicated_Images(YEAR, Filter_List_Bool: str=True):
    """
    Filter_List_Bool: If it is True, it means use image paths in Filter_List;
        Or use a whole year's worth of image paths in `f"{YEAR}_AD/"`.
    """
    def get_date_and_version(filepath):
        # Extract the date and the second number from the filename
        filename = os.path.basename(filepath)
        parts = filename.split('_')
        date_part = parts[0]  # 'YYYYMMDD'
        version = parts[1]  # '8', '7', '13', etc.
        return date_part, version
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"

    filter_image_paths = []
    outlier_image_paths = []
    if Filter_List_Bool:
        filter_Path = FOLD_PATH + f"{YEAR}_Filter_List.pkl"
        outier_Path = FOLD_PATH + f"{YEAR}_Outlier_List.pkl"
        if os.path.exists(filter_Path):
            with open(FOLD_PATH + f"{YEAR}_Filter_List.pkl", "rb") as f:
                filter_image_paths = pickle.load(f)
        else:
            print(f"{filter_Path} does not exist!")
            return

        if os.path.exists(outier_Path):
            with open(FOLD_PATH + f"{YEAR}_Outlier_List.pkl", "rb") as f:
                outlier_image_paths = pickle.load(f)
        else:
            print(f"{outier_Path} does not exist!")
            return
    else: 
        for filename in os.listdir(FOLD_PATH):
            file_path = os.path.join(FOLD_PATH, filename)
            name = filename.split(".")[0]
            suffix = filename.split(".")[1]
            # Check if it is a file (not a directory)
            if os.path.isfile(file_path): # Ensure it is a file
                if suffix == "png": # Ensure it is an image
                    if len(name.split('_')) == 5: # Ensure the image is an ad block
                        filter_image_paths.append(file_path)

    # Group images by date and version
    date_version_groups = defaultdict(list)

    for path in filter_image_paths:
        date_key, version_key = get_date_and_version(path)
        # Use both date and number as the key for grouping
        date_version_groups[(date_key, version_key)].append(path)

    Transfered_Paths = []
    # Display the images grouped by date and second number
    for (date, version), paths in date_version_groups.items():
        if len(paths) > 1:  # Check if there are more than 1 image in the group
            print(f"Date: {date}_{version}")
            Largest_Image_Path = paths[0]
            for path in paths:
                print(f"{path} with {(Get_File_Size(path) / 1024 ** 2):.3f} MB")
                Compare = Compare_File_Sizes(Largest_Image_Path, path)
                Largest_Image_Path = Compare if Compare else Largest_Image_Path
            print(f"Largest image: {Largest_Image_Path}")
            Transfered_Paths.append(Largest_Image_Path)
    
    Final_Filter, Final_Outlier = Update_File_Lists(filter_image_paths, outlier_image_paths, Transfered_Paths)
    print("*" * 50)
    print("Final Filter image paths:")
    for image in Final_Filter:
        print(image)
    print("Length:", len(Final_Filter))
    print("*" * 50)
    print("Final Outlier image paths:")
    for image in Final_Outlier:
        print(image)
    print("Length:", len(Final_Outlier))
    print("*" * 50)

    # Saving Filter list to a pickle file
    with open(FOLD_PATH + f'{YEAR}_Final_Filter.pkl', 'wb') as f:
        pickle.dump(Final_Filter, f)
        print(f"{FOLD_PATH + f'{YEAR}_Final_Filter.pkl'} saved.")

    # Saving Outlier list to a pickle file
    with open(FOLD_PATH + f'{YEAR}_Final_Outlier.pkl', 'wb') as f:
        pickle.dump(Final_Outlier, f)
        print(f"{FOLD_PATH + f'{YEAR}_Final_Outlier.pkl'} saved.")
    return Final_Filter, Final_Outlier
# A, B = Check_Duplicated_Images("2022", True)

In [16]:
def Modify_Chars(
    input_string,
    remove_chars_list=[
        " ",
        "@",
        "#",
        "^",
        "&",
        "*",
        "/",
        "\\",
        "+",
        "-",
        "=",
        "}",
        "{",
        "|",
        "~",
        "〉",
        "`",
        "〕",
        "「",
        "_",
        "'",
        "[",
        "]",
        "!",
        "<",
        ">",
        "ˇ",
        "′",
        "￥",
        "¥",
        "$"
    ],
    replace_dict={
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        ",": "，",
        ":": "：",
        ";": "；"
    },
):
    # Create a remove translation table with str.maketrans
    Remove = str.maketrans("", "", "".join(remove_chars_list))

    # Create a replace translation table
    Replace = str.maketrans(replace_dict)
    # Use str.translate to remove the specified characters
    return input_string.translate(Remove).translate(Replace)

In [17]:
def Json_Dict(Json_Path):
    """
    Json_Path: Path of a json file
    """
    with open(Json_Path, 'r', encoding='utf-8') as file:
        data_dict = json.load(file)  # Read and convert JSON to a dictionary
    return data_dict

def Dict_To_Json(Dict, Json_Path):
    """
    Dict: any dict format variable in python

    Json_Path: Path of a json file
    """
    with open(Json_Path, 'w', encoding='utf-8') as file:
        json.dump(Dict, file, ensure_ascii=False, indent=4)  # Write as formatted JSON

def FAD_Image(YEAR):
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    FAD_image_path = []
    for filename in os.listdir(FOLD_PATH):
        file_path = os.path.join(FOLD_PATH, filename)
        name = filename.split(".")[0]
        suffix = filename.split(".")[1]
        # Check if it is a file (not a directory)
        if os.path.isfile(file_path): # Ensure it is a file
            if suffix == "png": # Ensure it is an image
                if name.split('_')[2] == "FAD": # Ensure the image is FAD
                    FAD_image_path.append(file_path)
    # Save FAD path list to local
    FAD_PATH = FOLD_PATH + f"{YEAR}_FAD.pkl"
    with open(FAD_PATH, "wb") as f:
        pickle.dump(FAD_image_path, f)
        print(f"{YEAR}'s FAD are saved to {FAD_PATH}")
    return FAD_image_path

def Text_Recognition(YEAR, Begin_date="0101", End_date="1231"):
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))

    FAD_Path = FOLD_PATH + f"{YEAR}_FAD.pkl"
    if os.path.exists(FAD_Path):
        with open(FAD_Path, "rb") as f:
            FAD_Image_list = pickle.load(f)
    else:
        FAD_Image_list = FAD_Image(YEAR)

    # Load f"{YEAR}_Final_Filter.pkl"
    with open(FOLD_PATH + f"{YEAR}_Final_Filter.pkl", "rb") as f:
        final_filter_list = pickle.load(f)
    All_Image_list = list(set(FAD_Image_list).union(set(final_filter_list)))
    All_Image_list.sort()
    filtered_image_names = []
    for file_path in All_Image_list:
        filename = os.path.basename(file_path)
        name = filename.split(".")[0]
        suffix = filename.split(".")[1]
        # Check if it is a file (not a directory)
        if os.path.isfile(file_path): # Ensure it is a file
            if suffix == "png":
                if ((name.split('_')[2] == "FAD") 
                    or ("Block" in name.split('_'))): # Ensure the image is an ad block
                    date_str = name.split("_")[0]  # Get the date part
                    file_date = datetime.strptime(
                        date_str, "%Y%m%d")  # Convert to datetime object
                    # Check if the file date is within the specified range
                    if start_date <= file_date <= end_date:
                        filtered_image_names.append(filename)
                        # print(filename)
    # print(filtered_image_names)
    DATE_i= filtered_image_names[0].split("_")[0]
    print(DATE_i)
    for image_name in filtered_image_names:
        IMAGE_PATH = os.path.join(FOLD_PATH, image_name)
        NAME = image_name.split(".")[0]
        DATE = NAME.split("_")[0]
        json_path = FOLD_PATH + NAME + ".json"
        if DATE != DATE_i:  # This is used for date print
            print(DATE)
            DATE_i = DATE
        # Load an image from file
        if os.path.exists(json_path):
            print(f"Image text is stored in {json_path}")
            continue
        image = Image.open(IMAGE_PATH)  # Replace with your image file path

        # Perform OCR on the image
        extracted_text_E_C = pytesseract.image_to_string(image, lang="eng+chi_sim")
        extracted_text_C = pytesseract.image_to_string(image, lang="chi_sim")
        extracted_text_E_C = Modify_Chars(extracted_text_E_C)
        extracted_text_C = Modify_Chars(extracted_text_C)
        # Print the extracted text
        # print(extracted_text)

        text_dict = {}
        text_dict["Pytesseract_Text_E_C"] = extracted_text_E_C
        text_dict["Pytesseract_Text_E_C_Len"] = len(extracted_text_E_C)
        text_dict["Pytesseract_Text_C"] = extracted_text_C
        text_dict["Pytesseract_Text_C_Len"] = len(extracted_text_C)
        Dict_To_Json(text_dict, json_path)
        print(f"Image text is stored in {json_path}")
Text_Recognition("2022")
# FAD_Image("2022")
# 3~4 h

20220101
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220101_8_FAD.json
20220102
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220102_7_FAD.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220102_8_FAD.json
20220104
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220104_13_HAD_Block_2.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220104_14_HAD_Block_1.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220104_16_CV_Block_1.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220104_17_HAD_Block_1.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220104_19_FAD.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220104_8_FAD.json
20220105
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220105_12_CV_Block_1.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220105_14_CV_Block_1.json
Image text is stored in D:/AI_data_analysis/RMRB/2022_AD/20220105

In [ ]:
def Summarized_Text(YEAR, Begin_date="0101", End_date="1231", 
                    Model_Name="RMRB_model_2", Max_Len=1024,
                    Text_Type: str="Pytesseract_Text", 
                    Text_Language: list=["E_C", "C"]):
    """
    Hugging token (Name: RMRB_Analysis): ...
    Available models:
        1. RMRB_model_1: `uer/pegasus-base-chinese-cluecorpussmall`
        2. RMRB_model_2: `IDEA-CCNL/Randeng-Pegasus-238M-Summary-Chinese`
            # For model 2
            # Need to download tokenizers_pegasus.py and other Python script from Fengshenbang-LM github repo in advance,
            # or you can download tokenizers_pegasus.py and data_utils.py in https://huggingface.co/IDEA-CCNL/Randeng_Pegasus_523M/tree/main
            # Strongly recommend you git clone the Fengshenbang-LM repo:
            # 1. git clone https://github.com/IDEA-CCNL/Fengshenbang-LM
            # 2. cd Fengshenbang-LM/fengshen/examples/pegasus/
            # and then you will see the tokenizers_pegasus.py and data_utils.py which are needed by pegasus model
            #
            # Attention: the max length of Model_2 should be less than 1024, or pop up errors.
    All models saved to: D:/AI_data_analysis/RMRB/Text_Models
    """
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    model = PegasusForConditionalGeneration.from_pretrained(f"D:/AI_data_analysis/RMRB/Text_Models/{Model_Name}")
    tokenizer = PegasusTokenizer.from_pretrained(f"D:/AI_data_analysis/RMRB/Text_Models/{Model_Name}")

    filtered_json_names = []
    for filename in os.listdir(FOLD_PATH):
        file_path = os.path.join(FOLD_PATH, filename)
        name = filename.split(".")[0]
        suffix = filename.split(".")[1]
        # Check if it is a file (not a directory)
        if os.path.isfile(file_path):  # Ensure it is a file
            if suffix == "json": # Ensure it is json file
                date_str = name.split("_")[0]  # Get the date part
                file_date = datetime.strptime(
                    date_str, "%Y%m%d")  # Convert to datetime object
                # Check if the file date is within the specified range
                if start_date <= file_date <= end_date:
                    filtered_json_names.append(filename)

    DATE_i= filtered_json_names[0].split("_")[0]
    print(DATE_i)
    for json_name in filtered_json_names:
        json_path = os.path.join(FOLD_PATH, json_name)
        DATE = json_name.split("_")[0]
        if DATE != DATE_i:  # This is used for date print
            print(DATE)
            DATE_i = DATE
        # Checking if the JSON file exists
        if os.path.exists(json_path):
            # Load the existing data from the JSON file
            Json_Data_Dict = Json_Dict(json_path)
            for Lan in Text_Language:
                Summary_Text_Name = f"{Text_Type}_{Lan}_{Model_Name}_m{Max_Len}"
                if Summary_Text_Name not in Json_Data_Dict: # If the text has been summarized, pass it
                    TEXT = Json_Data_Dict[f"{Text_Type}_{Lan}"]
                    inputs = tokenizer(TEXT, max_length=Max_Len, return_tensors="pt")
                    # Generate Summary
                    summary_ids = model.generate(inputs["input_ids"])
                    summarized_text = tokenizer.batch_decode(
                        summary_ids, 
                        skip_special_tokens=True, 
                        clean_up_tokenization_spaces=False)[0]
                    # Modify the dictionary (for demonstration, you can add/modify/remove any keys)
                    Json_Data_Dict[Summary_Text_Name] = summarized_text
                    Dict_To_Json(Json_Data_Dict, json_path)
            print(f"{Text_Type}_{Lan} summarized by {Model_Name}_m{Max_Len} to {json_path}")
        else:
            print(f"{json_path} does not exist!")
Summarized_Text("2022")
# 1h

20220101
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220101_8_FAD.json
20220102
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220102_7_FAD.json
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220102_8_FAD.json
20220104
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220104_13_HAD_Block_2.json
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220104_14_HAD_Block_1.json
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220104_16_CV_Block_1.json
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220104_17_HAD_Block_1.json
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysis/RMRB/2022_AD/20220104_19_FAD.json
Pytesseract_Text_C summarized by RMRB_model_2_m1024 to D:/AI_data_analysi